In [ ]:
import os
import torch
import torch.nn as nn
from google.colab import drive
from transformers import AutoTokenizer, PreTrainedModel, PretrainedConfig
from transformers.modeling_outputs import CausalLMOutput
from transformers.generation import GenerationMixin

# load model from drive checkpoint
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

MODEL_DIR = "/content/drive/MyDrive/recipe_gpt2/model_minimal_250k_ingfirst"
BOS, EOS = "<|startofrecipe|>", "<|endofrecipe|>"

# match the code in the training
class MinimalGPTConfig(PretrainedConfig):
    model_type = "minimal_gpt"

    def __init__(self, vocab_size=50257, n_embd=512, n_layer=6, n_head=8,
                 n_inner=2048, ctx=512, **kwargs):
        self.vocab_size = vocab_size
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_inner = n_inner
        self.ctx = ctx

        self.num_hidden_layers = n_layer
        self.num_attention_heads = n_head
        self.hidden_size = n_embd
        self.max_position_embeddings = ctx
        super().__init__(**kwargs)


class MinimalGPTForCausalLM(PreTrainedModel, GenerationMixin):
    config_class = MinimalGPTConfig

    _tied_weights_keys = []
    all_tied_weights_keys = {}

    def __init__(self, config):
        super().__init__(config)

        self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.ctx, config.n_embd)
        layer = nn.TransformerEncoderLayer(
            d_model=config.n_embd,
            nhead=config.n_head,
            dim_feedforward=config.n_inner,
            activation="gelu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=config.n_layer)
        self.head = nn.Linear(config.n_embd, config.vocab_size)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        T = input_ids.shape[1]

        pos = torch.arange(T, device=input_ids.device)
        h = self.token_emb(input_ids) + self.pos_emb(pos)

        mask = nn.Transformer.generate_square_subsequent_mask(T).to(input_ids.device)
        h = self.transformer(h, mask=mask)
        logits = self.head(h)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = nn.functional.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )
        return CausalLMOutput(loss=loss, logits=logits)

    def prepare_inputs_for_generation(self, input_ids, **kwargs):
        return {"input_ids": input_ids[:, -self.config.ctx:]}


tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = MinimalGPTForCausalLM.from_pretrained(MODEL_DIR)
model.eval()
if torch.cuda.is_available():
    model.to("cuda")
print("Model loaded from", MODEL_DIR)

def make_recipe(ingredients, temperature=0.7, top_p=0.9, max_length=512):
    # Sort + lowercase to match training
    norm = ", ".join(sorted(i.strip().lower() for i in ingredients.split(",") if i.strip()))
    prompt = f"{BOS}<|ingredients|>{norm}<|title|>"   # ends at title -> model writes the rest
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            inputs["input_ids"], attention_mask=inputs["attention_mask"],
            max_length=max_length, do_sample=True, temperature=temperature, top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids(EOS),
        )
    text = tokenizer.decode(out[0], skip_special_tokens=False)
    print(text.replace(BOS, "").replace(EOS, "").replace("<|pad|>", "")
              .replace("<|ingredients|>", "INGREDIENTS: ")
              .replace("<|title|>", "\nTITLE: ")
              .replace("<|directions|>", "\nDIRECTIONS: ").strip())

[transformers] You are using a model of type `minimal_gpt` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded from /content/drive/MyDrive/recipe_gpt2/model_minimal_250k_ingfirst


In [9]:
make_recipe("chicken, mango, lime, tortillas")

INGREDIENTS: chicken, lime, mango, tortillas
TITLE: Chicken Tacos
DIRECTIONS: Cut chicken into bite size pieces. Put into large frying pan with the chicken pieces. Add the mango and lime juice. Cook on medium heat until the chicken is cooked through. Serve with lettuce, tomatoes, and avocado.


In [10]:
make_recipe("chicken, beef, pork")

INGREDIENTS: beef, chicken, pork
TITLE: Baked Pork
DIRECTIONS: Pour chicken broth over pork. Bake in a 350° oven for about 2 hours.


In [11]:
make_recipe("bacon, chili powder, chocolate, maple syrup, milk, sugar, watermelon")

INGREDIENTS: bacon, chili powder, chocolate, maple syrup, milk, sugar, watermelon
TITLE: Chili Con Queso
DIRECTIONS: Cut the watermelon into cubes. Cut the bacon into small pieces and fry in a little oil until brown. Place the watermelon cubes in a bowl with the syrup, chili powder, sugar and watermelon cubes. Pour in the milk and cook on low heat for 30 minutes.


In [12]:
make_recipe("rice, tofu, eggs, salt, pepper")

INGREDIENTS: eggs, pepper, rice, salt, tofu
TITLE: Tofu Filling
DIRECTIONS: Take a cup of rice and mix it with the tofu. Make a paste of the pepper and tofu. Put some salt, pepper and rice in a bowl. Put the sauce on the tofu and let it soak for at least 30 minutes. Put the sauce on top of the tofu. Then add some salt.


In [13]:
make_recipe("shrimp, lettuce, tomato, cucumber, rice, syrup")

INGREDIENTS: cucumber, lettuce, rice, shrimp, syrup, tomato
TITLE: Shrimp Salad
DIRECTIONS: Peel and devein shrimp. Mix in rice, tomato and shrimp. Add syrup and chill.
